# Telangana Crop Yield – Cleaning (single notebook)
Same steps as before, with one change: **Rain_Type cut-offs are built from train years (≤ 2015) only**, so nothing from the test years leaks in.

Input: `telangana_final.csv` → Output: `telangana_clean.csv`, `crop_season_lookup.csv`

Run the cells from top to bottom.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# ---------------- Settings ----------------
INPUT_FILE = "telangana_final.csv"
OUTPUT_FILE = "telangana_clean.csv"
LOOKUP_FILE = "crop_season_lookup.csv"

MIN_AREA_HA = 10         # tiny plots give noisy yields
IQR_FACTOR = 3           # outlier limit (per crop)
MIN_ROWS_PER_CROP = 30   # crops with fewer rows are removed
MIN_RAIN_YEARS = 6       # fewer train years than this -> use state-wide cut-offs
TRAIN_LAST_YEAR = 2015   # cut-offs are learned from years <= this only
DRY_Q, WET_Q = 1 / 3, 2 / 3


def log(step, df):
    print(f"{step:<45} rows left: {len(df)}")


df = pd.read_csv(INPUT_FILE)
log("0. Loaded file", df)

In [ ]:
# 1. Tidy text columns
df["District"] = df["District"].str.strip().str.title()
df["Season"] = df["Season"].str.strip().str.lower()
df["Crop"] = (df["Crop"].str.replace(r"\s+", " ", regex=True)
                        .str.replace("&M", "& M")
                        .str.strip())
df["Crop"] = df["Crop"].str[0].str.upper() + df["Crop"].str[1:]
df = df.drop(columns=["State"])

In [ ]:
# 2. Missing Production (Yield was a fake 0)
df = df.dropna(subset=["Production"])
log("2. Dropped missing Production", df)

In [ ]:
# 3. Hyderabad (placeholder rows)
df = df[df["District"] != "Hyderabad"]
log("3. Dropped Hyderabad", df)

In [ ]:
# 4. Same units (tonnes): Coconut dropped, Cotton & Mesta bales -> tonnes
df = df[df["Crop"] != "Coconut"]
df.loc[df["Crop"] == "Cotton(lint)", "Production"] *= 0.170
df.loc[df["Crop"] == "Mesta", "Production"] *= 0.180
log("4. Dropped Coconut, bales -> tonnes", df)

In [ ]:
# 5. Merge duplicate keys (add Area & Production, keep one weather copy)
key = ["District", "Crop", "Crop_Year", "Season"]
weather = ["rain_mm", "rainy_days", "temp_mean", "temp_max", "temp_min", "hot_days_35", "et0_mm"]
rules = {"Area": "sum", "Production": "sum"}
rules.update({c: "first" for c in weather})
df = df.groupby(key, as_index=False).agg(rules)
log("5. Merged duplicate rows", df)

In [ ]:
# 6. Recalculate Yield (tonnes per hectare)
df["Yield"] = (df["Production"] / df["Area"]).round(3)

In [ ]:
# 7. Remove bad rows
df = df[df["Area"] >= MIN_AREA_HA]
log("7a. Dropped very small areas", df)
df = df[df["Yield"] > 0]
log("7b. Dropped zero-yield rows", df)

In [ ]:
# 8. Yield outliers, judged per crop
g = df.groupby("Crop")["Yield"]
q1 = g.transform(lambda s: s.quantile(0.25))
q3 = g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
size = g.transform("count")
outlier = ((df["Yield"] < q1 - IQR_FACTOR * iqr) | (df["Yield"] > q3 + IQR_FACTOR * iqr)) & (size >= 20)
df = df[~outlier]
log("8. Dropped yield outliers (per crop)", df)

In [ ]:
# 8b. Crops with too few rows
crop_rows = df["Crop"].map(df["Crop"].value_counts())
removed = sorted(df.loc[crop_rows < MIN_ROWS_PER_CROP, "Crop"].unique())
df = df[crop_rows >= MIN_ROWS_PER_CROP]
log("8b. Dropped crops with too few rows", df)
print("    crops removed:", ", ".join(removed))

In [ ]:
# 9. Rain_Type (dry / normal / wet) - LEAK-FREE
#    Rain is the same for every crop in a district-year-season,
#    so work on one row per district-year-season.
rain = df[["District", "Season", "Crop_Year", "rain_mm"]].drop_duplicates(["District", "Season", "Crop_Year"])
train_rain = rain[rain["Crop_Year"] <= TRAIN_LAST_YEAR]          # cut-offs learn from these years ONLY

# Way 1: each district's own dry/wet cut-offs (train years only)
own = (train_rain.groupby(["District", "Season"])["rain_mm"]
       .agg(n_years="count", dry_cut=lambda s: s.quantile(DRY_Q), wet_cut=lambda s: s.quantile(WET_Q))
       .reset_index())
own = own[own["n_years"] >= MIN_RAIN_YEARS].drop(columns="n_years")

# Way 2: state-wide cut-offs per season (train years, districts with enough years)
enough = train_rain.merge(own[["District", "Season"]], on=["District", "Season"])
state = (enough.groupby("Season")["rain_mm"]
         .agg(state_dry=lambda s: s.quantile(DRY_Q), state_wet=lambda s: s.quantile(WET_Q))
         .reset_index())

rain = rain.merge(own, on=["District", "Season"], how="left").merge(state, on="Season", how="left")
has_own = rain["dry_cut"].notna()
rain["rain_type_source"] = np.where(has_own, "own district", "state-wide")
dry_cut = rain["dry_cut"].fillna(rain["state_dry"])
wet_cut = rain["wet_cut"].fillna(rain["state_wet"])
rain["Rain_Type"] = np.select([rain["rain_mm"] <= dry_cut, rain["rain_mm"] > wet_cut],
                              ["dry", "wet"], default="normal")

df = df.merge(rain[["District", "Season", "Crop_Year", "Rain_Type", "rain_type_source"]],
              on=["District", "Season", "Crop_Year"])
print("\nRain_Type by method:")
print(df.groupby(["rain_type_source", "Rain_Type"]).size().unstack().to_string())

In [ ]:
# 10. Target: log_yield  (convert back with np.expm1)
df["log_yield"] = np.log1p(df["Yield"])

In [ ]:
# 11. Final checks
assert df.isna().sum().sum() == 0, "missing values left"
assert not df.duplicated(key).any(), "duplicate keys left"
assert (df["Yield"] > 0).all(), "non-positive yield left"
assert np.isfinite(df.select_dtypes("number")).all().all(), "infinite values left"

In [ ]:
# 12. Save
df = df.sort_values(["Crop_Year", "District", "Crop", "Season"]).reset_index(drop=True)
df.to_csv(OUTPUT_FILE, index=False)
df.groupby(["Crop", "Season"]).size().rename("rows").reset_index().to_csv(LOOKUP_FILE, index=False)

print(f"\nSaved {len(df)} rows, {df['Crop'].nunique()} crops, {df['District'].nunique()} districts -> {OUTPUT_FILE}")
print("\nRain_Type counts:")
print(df["Rain_Type"].value_counts().to_string())